# 041 数据清洗：评论数据

In [ ]:
TOPIC_COMMENT_PATH = r'..\data\fe\topic_comment.parquet'

In [ ]:
import pandas as pd
import re

# ========== 加载原始数据 ==========
df_topic_comment = pd.read_parquet(TOPIC_COMMENT_PATH)


# 记录表原始行数，用于全程追踪保留率
_ORIGINAL_COUNTS = {
    "topic_comment": len(df_topic_comment),
}

def retention_rate(table: str, current_df) -> str:
    """计算相对于原始数据的保留率。"""
    orig = _ORIGINAL_COUNTS[table]
    cur = len(current_df)
    return f"{cur:,} / {orig:,} ({cur / orig * 100:.1f}% 保留)"

print("\n📦 原始数据量:")
print(f"  topic_comment: {len(df_topic_comment):>10,}")

In [ ]:
def clean_text(text: str, type: str) -> str:
    """清洗微博评论文本，保留情绪信号。

    清洗步骤（有序）：
    1. 移除 HTML 标签
    2. 移除 URL
    3. 移除 回复@xxx: 前缀（保留正文部分）
    4. 规范化空白字符
    """
    if not isinstance(text, str) or len(text) == 0:
        return text

    # 1. 移除 HTML 标签
    text = re.sub(r'<[^>]+>', '', text)

    # 2. 移除 URL
    text = re.sub(r'https?://\S+', '', text)

    # 3. 移除 "回复@xxx：" 或 "回复@xxx:" 前缀，保留后续正文
    if type == "comment":
        text = re.sub(r'^回复@[\w\u4e00-\u9fff]+[：:]', '', text)

    # 4. 话题标签：#xxx# → xxx（保留标签文字内容）
    # text = re.sub(r'#([^#]+)#', r'\1', text)

    # 5. 规范化空白（多个空白合并为一个，去首尾空白）
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# ========== 应用文本清洗 ==========
# 应用清洗
df_topic_comment["content"] = df_topic_comment["content"].apply(clean_text, type="comment")

In [ ]:
# ========== 3.5 topic_comment 文本质量五级标注 ==========
# Level 0: 空文本（空字符串 / 纯空白 / 换行）
_L0_EMPTY = re.compile(r'^\s*$')

# Level 1: 信息量极低（模板转发、纯重复字符、纯数字/符号/Emoji/英文、纯@用户）
_L1_TEMPLATE = re.compile(
    r'^([转轉][发發]微博|图片评论( 评论配图)?|评论配图|哈+|啊+)$'
)
_L1_DIGITS   = re.compile(r'^\d+$')
_L1_SYMBOLS  = re.compile(r'^[^\w\u4e00-\u9fff\U00010000-\U0010FFFF]+$')
_L1_ALPHA    = re.compile(r'^[a-zA-Z]+$')
_L1_EMOJI    = re.compile(
    r'^[\U0001F300-\U0001F9FF\U00002600-\U000027BF'
    r'\U0001FA00-\U0001FA6F\U0001FA70-\U0001FAFF\uFE00-\uFE0F\u200D]+$'
)
_L1_AT_ONLY  = re.compile(r'^(@[\u4e00-\u9fa5a-zA-Z0-9_-]+\s*)+$') 

# Level 2: 有基本语义但情绪分析价值低（寒暄、礼貌互动等）
# 原始规则（保留以兼容性）
_L2_GREET_LEGACY = re.compile(
    r'^(感谢分享(精彩内容)?|谢谢(分享|你的支持)?|晚安(网页链接)?'
    r'|早+(安|上好([哦呀啊哇])?)?'
    r'|([上中下]午好)([哦呀啊哇])?'
    r'|晚(安|上好([哦呀啊哇])?)?'
    r'|新年(快乐|好)|关注|持续关注'
    r'|周(一|二|三|四|五|六|日|末)愉快'
    r'|了解(一下|了)?|签到|确认签收|接(好运|接接)?'
    r'|是的(呢)?|是(啊|呀|的呢?|这样的|的)|对+(的|啊)?|说的对'
    r'|没毛病|有道理|嗯(嗯|呢|呐)?|我也觉得|我也是'
    r'|就是(就是|啊|的)?|确实(是这样)?|不错(不错)?|可以'
    r'|赞(同)?|顶|up|哇|啊|哎|来了|好的|看看|好家伙'
    r'|分享知识传播正能量)$',
    re.IGNORECASE
)

# ===== 新增规则集合（R2.1, R2.3, R2.4, R2.5）=====

# R2.1: 纯寒暄/问候（带对象/称呼/语气词/祝福词）
# 规则：包含核心问候词 + 长度<=30字 + 无实质观点词
_R21_GREET_WORDS = r'(早(上|安|上好)|晚(上好|安)|周[一二三四五六日末]愉快)'
# 黑名单：实质性观点词、讨论词，不包括寒暄性的"加油"、"祝福"等
_R21_NO_SUBSTANCE = r'(应该|必须|一定|拜拜|学到|厉害|棒|很|太|很好|真的)'

def _check_r21_pure_greet(text: str) -> bool:
    """R2.1: 纯寒暄/问候（可能带对象、称呼、语气词、祝福词，但无实质观点）"""
    if len(text) > 30:
        return False
    # 必须包含核心问候词
    if not re.search(_R21_GREET_WORDS, text):
        return False
    # 不能包含实质观点词
    if re.search(_R21_NO_SUBSTANCE, text):
        return False
    return True

# R2.3: 简单附和/确认（<=4字的纯附和词）
_R23_AFFIRMATION = re.compile(
    r'^(就是|对啊|没错|是啊|支持|同意|赞|赞同|好的|嗯|嗯呢|呀|确实是|没毛病|是|有|哦)(啊|呀|呢)?$',
    re.IGNORECASE
)

# R2.4: 礼貌互动/感谢（感谢+分享/提醒，长度<=30字）
# 规则1：感谢词 + 分享/提醒类对象
_R24_THANKS_PATTERN1 = re.compile(
    r'^(谢谢|感谢|多谢)(.*?)(分享|提醒|科普|支持)([啊呀哇哦很呢])?$',
    re.IGNORECASE
)
# 规则2：分享/提醒后面接感谢词（倒序）
_R24_THANKS_PATTERN2 = re.compile(
    r'^(分享|提醒|科普)(.*?)?(谢谢|感谢)([啊呀呢哇])?$',
    re.IGNORECASE
)

def _check_r24_thanks(text: str) -> bool:
    """R2.4: 礼貌互动/感谢（感谢分享型、分享感谢型）"""
    if len(text) > 30:
        return False
    return bool(_R24_THANKS_PATTERN1.search(text) or _R24_THANKS_PATTERN2.search(text))

# R2.5: 仪式性短语（纯单一仪式动词，无修饰）
# 包括：接好运、打卡、签到、接、到、来了、围观、支持
_R25_RITUAL_SINGLE = re.compile(
    r'^(接(好运)?|打卡|签到|来|到|来了|围观)$',
    re.IGNORECASE
)

def _check_r25_ritual(text: str) -> bool:
    """R2.5: 仪式性短语（接好运、打卡、签到等）"""
    return bool(_R25_RITUAL_SINGLE.search(text))

# R2.6: 问候+感谢混合型（新增）
# 规则：同时包含问候词和感谢词，长度<=40字，无实质观点
_R26_GREET_AND_THANKS = re.compile(
    r'(周[一二三四五六日末]愉快|早(上|安)|晚(安|上好)).*?(谢谢|感谢|感激|分享)',
    re.IGNORECASE
)

def _check_r26_greet_thanks(text: str) -> bool:
    """R2.6: 问候+感谢混合型（如'周末愉快，感谢分享'）"""
    if len(text) > 40:
        return False
    # 必须同时包含问候词和感谢词
    has_greet = re.search(r'(周[一二三四五六日末]愉快|早(上|安)|晚(安|上好))', text)
    has_thanks = re.search(r'(谢谢|感谢|感激|分享)', text)
    if not (has_greet and has_thanks):
        return False
    # 不能包含"加油"等实质期盼词（这些应该保留L3）
    if re.search(r'(加油|必须|应该|一定)', text):
        return False
    return True


def assign_text_quality(text: str) -> int:
    """为 topic_comment 的 content 字段分配文本质量等级（0-4）。

    新增规则（Phase 1 + 增强）：
    - R2.1: 纯寒暄/问候（可能带对象/称呼/语气词，但无实质观点）- 长度<=30字
    - R2.3: 简单附和/确认（<=4字的纯附和词）
    - R2.4: 礼貌互动/感谢（感谢分享型、分享感谢型，长度<=30字）
    - R2.5: 仪式性短语（接好运、打卡、签到等）
    - R2.6: 问候+感谢混合型（如'周末愉快，感谢分享'，长度<=40字）

    Returns:
        `int`:
            文本质量等级：
            - 0: 空文本
            - 1: 信息量极低（模板/纯数字/纯符号/纯Emoji/纯英文）
            - 2: 低分析价值（寒暄/礼貌互动/仪式性）
            - 3: 可分析（默认，有语义和态度线索）
            - 4: 高分析价值（由后续长度阈值提升，此处不处理）
    """
    if not isinstance(text, str):
        return 0
    t = text.strip()
    
    # Level 0: 空文本
    if _L0_EMPTY.fullmatch(t):
        return 0
    
    # Level 1: 信息量极低
    if (
        _L1_TEMPLATE.fullmatch(t)
        or _L1_DIGITS.fullmatch(t)
        or _L1_SYMBOLS.fullmatch(t)
        or _L1_ALPHA.fullmatch(t)
        or _L1_EMOJI.fullmatch(t)
        or _L1_AT_ONLY.fullmatch(t)
    ):
        return 1
    
    # Level 2: 低分析价值
    # 先检查新增规则（R2.1, R2.3, R2.4, R2.5, R2.6）
    if _check_r21_pure_greet(t):
        return 2
    if _R23_AFFIRMATION.fullmatch(t):
        return 2
    if _check_r24_thanks(t):
        return 2
    if _check_r25_ritual(t):
        return 2
    if _check_r26_greet_thanks(t):
        return 2
    # 再检查原始规则（向下兼容）
    if _L2_GREET_LEGACY.fullmatch(t):
        return 2
    
    # Level 3: 默认可分析
    return 3


_QUALITY_LABELS = {0: "空文本", 1: "极低信息", 2: "低分析价值", 3: "可分析", 4: "高分析价值"}

df_topic_comment["text_quality"] = df_topic_comment["content"].apply(assign_text_quality)
df_topic_comment["text_quality_label"] = df_topic_comment["text_quality"].map(_QUALITY_LABELS)

# 打印各等级分布
print(f"\n✅ topic_comment 文本质量标注完成:")
quality_counts = df_topic_comment["text_quality"].value_counts().sort_index()
for level, count in quality_counts.items():
    label = _QUALITY_LABELS[level]
    print(f"  Level {level} ({label}): {count:,} 条 ({count / len(df_topic_comment) * 100:.2f}%)")

print(f"\n📊 当前数据量:")
print(f"  topic_comment: {len(df_topic_comment):>10,}")

In [ ]:
import os

# 创建输出目录
output_dir = r"..\data\cleaned"
os.makedirs(output_dir, exist_ok=True)

# 保存
datasets = {
    "topic_comment": df_topic_comment,
}

for name, df in datasets.items():
    path = os.path.join(output_dir, f"{name}.parquet")
    df.to_parquet(path, index=False)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"✅ {name}.parquet 已保存 ({df.shape[0]:>10,} rows × {df.shape[1]:>2} cols, {size_mb:.1f} MB)")

print(f"\n📂 输出目录: {os.path.abspath(output_dir)}")